![mental_health_banner_1774025040503.jpg](../mental_health_banner_1774025040503.jpg "mental_health_banner_1774025040503.jpg")

### Transformacion 

Notebook Transformacion de Datos para obtener estilo de vida de cada paciente

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalogo", "health_db_catalog")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_patients = spark.table(f"{catalogo}.{esquema_source}.patients_information")
df_state_codes = spark.table(f"{catalogo}.{esquema_source}.us_state_codes")
df_daily_nutrition = spark.table(f"{catalogo}.{esquema_source}.daily_nutrition_facts")
df_healthy_life = spark.table(f"{catalogo}.{esquema_source}.healthy_lifestyle")

In [0]:
from pyspark.sql.types import StructType, StructField, LongType
from pyspark.sql.functions import *

schema_of_json = StructType([
    StructField("Calcium", LongType(), True),
    StructField("Folate", LongType(), True),
    StructField("Iron", LongType(), True),
    StructField("Magnesium", LongType(), True),
    StructField("vitaminA", LongType(), True),
    StructField("vitaminB12", LongType(), True),
    StructField("vitaminC", LongType(), True),
    StructField("vitaminD", LongType(), True),
    StructField("vitaminE", LongType(), True),
    StructField("vitaminK", LongType(), True)
])

df_nutrition_transformed = df_daily_nutrition.select(
    col("person_id"),
    from_json(col("dialy_vitamins"), schema_of_json).alias("vitamins")
)

df_nutrition_final = df_nutrition_transformed.select(
    col("person_id"),
    col("vitamins.Calcium").alias("Calcium"),
    col("vitamins.Folate").alias("Folate"),
    col("vitamins.Iron").alias("Iron"),
    col("vitamins.Magnesium").alias("magnesium"),
    col("vitamins.vitaminA").alias("vitaminA"),
    col("vitamins.vitaminB12").alias("vitaminB12"),
    col("vitamins.vitaminC").alias("vitaminC"),
    col("vitamins.vitaminD").alias("vitaminD"),
    col("vitamins.vitaminE").alias("vitaminE"),
    col("vitamins.vitaminK").alias("vitaminK")
)

In [0]:
df_lifestyle_health = (
    df_patients.alias("pi").join(df_nutrition_final.alias("dn"), col("pi.person_id") == col("dn.person_id"),"left")
    .join(df_healthy_life.alias("hl"), col("pi.person_id") == col("hl.person_id"),"left")
    .join(df_state_codes.alias("sc"), col("pi.state") == col("sc.id"),"left")
    .select(col("pi.person_id")
           ,col("sc.state")
           ,col("pi.Gender")
           ,col("pi.Age")
           ,col("pi.Occupation")
           ,col("dn.VitaminD")
           ,col("dn.VitaminB12")
           ,col("dn.VitaminC")
           ,col("dn.VitaminA")
           ,col("dn.VitaminE")
           ,col("dn.vitaminK")
           ,col("dn.Folate")
           ,col("dn.Iron")
           ,col("dn.Calcium")
           ,col("dn.Magnesium")
           ,col("hl.physical_activity_level")
    )
)

In [0]:
df_lifestyle_health.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.lifestyle_habits")